# 02 — Compare classical regression models

In this notebook I run the package's classical regression experiment. The workflow keeps the model-fitting data, conformal-calibration data and final test data separate. Cross-validation is run only inside the model-fitting partition. The scale-sensitive estimators include their own `StandardScaler` pipeline, so each cross-validation training fold learns its scaling without using the fold's validation rows. The final test set is not used for model fitting or selection.

In [ ]:
from pathlib import Path

from diabetes_ml.experiment import run_classical_experiment

repo_root = Path.cwd()
if not (repo_root / "pyproject.toml").is_file():
    repo_root = repo_root.parent
if not (repo_root / "pyproject.toml").is_file():
    raise RuntimeError("Run this notebook from the repository root or notebooks directory.")
output_dir = repo_root / "notebook-results" / "classical"
metrics = run_classical_experiment(output_dir=output_dir, random_state=42)
display(metrics)
print(f"Saved experiment outputs to: {output_dir.resolve()}")

In [ ]:
import pandas as pd

saved_metrics = pd.read_csv(output_dir / "classical_metrics.csv")
display(saved_metrics)
assert saved_metrics.equals(metrics), "Saved metrics differ from the returned results."

## Reading the results

The cross-validation MAE summarises error across folds within the model-fitting partition. Test MAE and RMSE measure prediction error on the held-out test observations, in the target's units. Test R² can be negative when the model performs worse than predicting the test-set mean.

The conformal coverage and interval width describe the 90% split-conformal intervals on the held-out observations. The subgroup MAE comparison looks only at the two coded values of the source `sex` feature; it is an exploratory check, not a full fairness assessment.

The output folder also contains `fairness_audit.json`, `ridge_feature_attribution.csv` and `rf_feature_importance.csv`. These are summaries of the fitted models. They should not be interpreted as causal explanations or evidence of clinical validity.